In [1]:
%load_ext autoreload
%autoreload 2

In [115]:
import pathlib

import catboost as cb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from loguru import logger
from sklearn import compose, decomposition, linear_model, metrics, pipeline, preprocessing
from sklearn.model_selection import StratifiedGroupKFold

import highres_ta as hta

In [163]:
FoldIndices = tuple[np.ndarray, np.ndarray]


def load_data(glob_path_to_files:str = "../data/training/*.pq") -> pd.DataFrame:
    """
    Load all parquet files in a folder into a single pandas DataFrame.
    """
    
    # Get a list of all parquet files in the folder
    path = pathlib.Path(glob_path_to_files)
    parent_folder = path.parent
    parquet_files = list(parent_folder.glob(path.name))
    n_files = len(parquet_files)

    logger.debug(f"Loading {n_files} {path.suffix} files from {parent_folder}")

    # Load each parquet file into a DataFrame and concatenate them
    df_list = [pd.read_parquet(file) for file in parquet_files]
    combined_df = pd.concat(df_list, ignore_index=True)

    return combined_df


def load_talk_adjustment(fname: str) -> pd.Series:
    """
    Downloaded from : https://glodapv2.geomar.de/adjustments/select_cruises
    Should be able to use this directly
    """
    return (
        pd
        .read_csv(
            fname,
            skiprows=2,
            skipinitialspace=True,
            na_values=[-999, -888, -777, 0],
            usecols=["cruise_expocode", "alkalinity_adj"],
            index_col="cruise_expocode",
        )
        .dropna()
        .alkalinity_adj
    )


def add_talk_adjustment(df: pd.DataFrame, fname: str) -> pd.DataFrame:
    """
    Add a new column to the DataFrame with talk adjustments based on expocode.
    
    Parameters:
    - df: pd.DataFrame containing the data.
    - fname: Path to the CSV file containing talk adjustments.
    
    Returns:
    - pd.DataFrame with an additional column 'talk_adj' containing the adjustments.
    """
    
    adjustments = load_talk_adjustment(fname).to_dict()
    df = df.copy()
    df["talk_adj"] = df["expocode"].map(adjustments).fillna(0)
    
    return df


def make_salinity_bins(salinity: pd.Series, bins: list[float]|float=0.2, **cut_kwargs)-> pd.Series:
    """
    Create salinity bins from a pandas Series of salinity values.
    
    Parameters:
    - salinity: pd.Series containing salinity values.
    - bins: different behaviors depending on the type:
        - list of floats: use these values as bin edges.
        - int: create this many equal-width bins between the min and max salinity.
        - float: create bins with this percentile width (e.g., 0.2 for 20% width).
    
    Returns:
    - pd.Series with the same index as the input, containing the bin labels.
    """

    if isinstance(bins, list):
        # Use the provided list of bin edges
        bin_edges = bins
    elif isinstance(bins, (float, int)) and (bins > 1) and not (bins % 1):
        # must be an integer greater than 1, create equal-width bins
        bins = int(bins)
        bin_edges = np.linspace(salinity.min(), salinity.max(), bins + 1)
    elif isinstance(bins, float) and (0 < bins < 1) and (float(1 / bins)).is_integer():
        # Create bins based on percentiles
        quantiles = np.arange(0, 1 + bins/2, bins)
        if quantiles[0] != 0.0:
            raise ValueError("For percentile bins, the first quantile must be 0.0. Adjust the bins value accordingly.")
        if quantiles[-1] != 1.0:
            raise ValueError("For percentile bins, the last quantile must be 1.0. Adjust the bins value accordingly.")
        bin_edges = salinity.quantile(quantiles).values
    else:
        raise ValueError("bins must be a list of floats, an int[1:inf], or a float[0:1]")

    logger.debug(f"Using the following bin edges for salinity: {bin_edges}")

    # Create the bins and return the binned series
    cut_default_kwargs = {"include_lowest": True, "right": True, "labels": False}
    cut_passed_kwargs = cut_default_kwargs | cut_kwargs
    binned_salinity = pd.cut(salinity, bins=bin_edges, **cut_passed_kwargs)
    
    return binned_salinity


def add_salinity_bins(df: pd.DataFrame, salinity_col_name: str = "salinity", bins: list[float]|float = 0.2, **cut_kwargs) -> pd.DataFrame:
    """
    Add a new column to the DataFrame with salinity bins.
    
    Parameters:
    - df: pd.DataFrame containing the data.
    - salinity_col_name: Name of the column in df that contains salinity values.
    - bins: different behaviors depending on the type:
        - list of floats: use these values as bin edges.
        - int: create this many equal-width bins between the min and max salinity.
        - float: create bins with this percentile width (e.g., 0.2 for 20% width).
    
    Returns:
    - pd.DataFrame with an additional column 'salinity_bin' containing the bin labels.
    """
    
    df = df.copy()
    df["salinity_bin"] = make_salinity_bins(df[salinity_col_name], bins=bins, **cut_kwargs)
    
    return df


def stratified_group_folds(
    data: pd.DataFrame,
    *,
    stratify_by: str = "salinity_bin",
    group_by: str = "expocode",
    n_splits: int = 5,
    shuffle: bool = True,
    random_state: int | None = 42,
) -> list[FoldIndices]:
    """Return train/validation positional indices for stratified group CV."""

    logger.debug(f"Making train-test splits stratified by {stratify_by} and grouped by {group_by}")
    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=shuffle,
        random_state=random_state if shuffle else None,
    )

    return list(
        splitter.split(
            X=data,
            y=data[stratify_by],
            groups=data[group_by],
        )
    )


def make_train_test_folds(
    df: pd.DataFrame,
    *,
    expocode_col_name: str = "expocode",
    salinity_col_name: str = "salinity",
    salinity_bins: list[float]|float = 0.2,
    n_splits: int = 5,
    shuffle: bool = True,
    random_state: int | None = 42,
) -> list[FoldIndices]:
    """Return train/test positional indices for stratified group CV."""

    df = df.copy()
    df["salinity_bin"] = make_salinity_bins(df[salinity_col_name], bins=salinity_bins)

    return stratified_group_folds(
        data=df.reset_index(allow_duplicates=True, drop=False),
        stratify_by="salinity_bin",
        group_by=expocode_col_name,
        n_splits=n_splits,
        shuffle=shuffle,
        random_state=random_state,
    )


def make_train_test_split(
    df: pd.DataFrame,
    *,
    expocode_col_name: str = "expocode",
    salinity_col_name: str = "salinity",
    salinity_bins: list[float]|float = 0.2,
    n_splits: int = 5,
    with_validation: bool = False,
    shuffle: bool = True,
    random_state: int | None = 42,
    fold_index: int = 0,
) -> list[np.ndarray]:
    """Return train/test positional indices for stratified group CV."""

    folds = make_train_test_folds(
        df,
        expocode_col_name=expocode_col_name,
        salinity_col_name=salinity_col_name,
        salinity_bins=salinity_bins,
        n_splits=n_splits,
        shuffle=shuffle,
        random_state=random_state,
    )

    i = fold_index % n_splits
    if with_validation:
        train, test = folds[i]
        valid = folds[(i + 1) % n_splits][-1]
        train = np.setdiff1d(train, valid)
        folds = [train, test, valid]
    else:
        train, test = folds[i]
        folds = [train, test]

    unique = set()
    for f in folds:
        unique.update(f)
    if len(unique) != len(data):
        raise ValueError("Train/test/validation splits do not cover all data points.")

    # Return only the first fold as train/test split
    return folds


def latlon_to_spherical_coords(lat: pd.Series, lon: pd.Series) -> pd.DataFrame:
    """
    Convert latitude and longitude to spherical coordinates (x, y, z).
    
    Parameters:
    - lat: pd.Series of latitudes in degrees.
    - lon: pd.Series of longitudes in degrees.
    
    Returns:
    - Tuple of three pd.Series: (x, y, z) coordinates.
    """
    # Convert degrees to radians
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)

    # Calculate spherical coordinates
    n_coords = pd.DataFrame({
        "ncoord_x": pd.Series(np.cos(lat_rad) * np.cos(lon_rad), name="ncoord_x"),
        "ncoord_y": pd.Series(np.cos(lat_rad) * np.sin(lon_rad), name="ncoord_y"),
        "ncoord_z": pd.Series(np.sin(lat_rad), name="ncoord_z"),
    })

    return n_coords


def time_to_cyclical_dayofyear(time: pd.Series) -> pd.DataFrame:
    """
    Convert time to cyclical day-of-year features (sin and cos).
    
    Parameters:
    - time: pd.Series of datetime objects.
    
    Returns:
    - pd.DataFrame with two columns: 'dayofyear_sin' and 'dayofyear_cos'.
    """
    # Extract day of year
    day_of_year = time.dt.dayofyear

    # Calculate sine and cosine transformations
    dayofyear_sin = np.sin(2 * np.pi * day_of_year / 365.25)
    dayofyear_cos = np.cos(2 * np.pi * day_of_year / 365.25)

    return pd.DataFrame({
        "dayofyear_sin": dayofyear_sin,
        "dayofyear_cos": dayofyear_cos
    })


def make_linear_pipeline(linear_vars: list[str]) -> pipeline.Pipeline:
    """
    Create a linear regression pipeline for specified linear variables.

    Parameters:
    - linear_vars: List of variable names to include in the linear regression.

    Returns:
    - A scikit-learn Pipeline object.
    """
    linear_vars_pattern = r"^(?:" + "|".join(linear_vars) + r")$"

    return pipeline.make_pipeline(
        compose.make_column_transformer(
            ("passthrough", compose.make_column_selector(pattern=linear_vars_pattern)),
            remainder="drop",
            verbose_feature_names_out=False,
        ),
        preprocessing.StandardScaler(),
        decomposition.PCA(None),
        linear_model.LinearRegression(),
    )


def drop_extreme_salinities(df, salinity_col_name: str = "salinity", min: float = 20, max: float = 40):
    """
    Drop rows from the DataFrame where salinity is outside the specified range.

    Parameters:
    - df: pd.DataFrame containing the data.
    - salinity_col_name: Name of the column in df that contains salinity values.
    - min: Minimum acceptable salinity value (inclusive).
    - max: Maximum acceptable salinity value (inclusive).

    Returns:
    - pd.DataFrame with rows outside the specified salinity range removed.
    """
    return df[(df[salinity_col_name] >= min) & (df[salinity_col_name] <= max)]


def add_spherical_coords(df: pd.DataFrame, lat_col_name: str = "lat", lon_col_name: str = "lon") -> pd.DataFrame:
    """
    Add spherical coordinates (x, y, z) to the DataFrame based on latitude and longitude.

    Parameters:
    - df: pd.DataFrame containing the data.
    - lat_col_name: Name of the column in df that contains latitude values.
    - lon_col_name: Name of the column in df that contains longitude values.

    Returns:
    - pd.DataFrame with additional columns 'ncoord_x', 'ncoord_y', and 'ncoord_z'.
    """
    df_all = df.copy().reset_index()
    spherical_coords = latlon_to_spherical_coords(df_all[lat_col_name], df_all[lon_col_name]).set_index(df.index)
    return df.assign(**spherical_coords)


def add_cyclical_dayofyear(df: pd.DataFrame, time_col_name: str = "time") -> pd.DataFrame:
    """
    Add cyclical day-of-year features (sin and cos) to the DataFrame based on a time column.

    Parameters:
    - df: pd.DataFrame containing the data.
    - time_col_name: Name of the column in df that contains datetime values.

    Returns:
    - pd.DataFrame with additional columns 'dayofyear_sin' and 'dayofyear_cos'.
    """
    df_all = df.copy().reset_index(drop=False)
    cyclical_features = time_to_cyclical_dayofyear(df_all[time_col_name]).set_index(df.index)

    return df.assign(**cyclical_features)


def drop_bad_quality_talk(df: pd.DataFrame, talkf_col_name: str = "talkf", talk_adj_col_name: str = "talk_adj") -> pd.DataFrame:
    """
    Drop rows from the DataFrame where the quality control columns indicate bad quality.

    Parameters:
    - df: pd.DataFrame containing the data.
    - talkf_col_name: Name of the column in df that contains additional quality flags.
    - talk_adj_col_name: Name of the column in df that contains talk adjustments.

    Returns:
    - pd.DataFrame with rows indicating bad quality removed.
    """
    adjustment = df[talk_adj_col_name].abs()
    adjustment_thresh = 6
    small_adjustment = adjustment <= adjustment_thresh
    good_flags = df[talkf_col_name] == 2
    
    large_adjustment_count = (~small_adjustment).sum()
    notgood_flags_count = (~good_flags).sum()
    filtered_count = (~(good_flags & small_adjustment)).sum()
    total_count = len(df)

    logger.debug(f"TA values with large adjustments (<= {adjustment_thresh}): {large_adjustment_count}")
    logger.debug(f"TA values without good flags (!= 2): {notgood_flags_count}")
    logger.info(f"Number of rows filtered due to large adjustments and bad flags: {filtered_count} of {total_count} ({filtered_count / total_count:.0%})")

    return df[good_flags & small_adjustment]

In [195]:
COORD_COLUMNS = [
    "expocode",
    "time",
    "lat",
    "lon",
    "depth",
]
TARGET_NAME = "talk"
FEATURE_NAMES = [
    "bottomdepth",
    "temperature",
    "salinity",
    "nitrate",
    "ssh_adt",
    # "ssh_sla",
]
QC_COLS = ["talkqc", "talkf"]
REQUIRED_COLUMNS = list(set(COORD_COLUMNS + FEATURE_NAMES + [TARGET_NAME] + QC_COLS))
NONAN_SUBSET = [TARGET_NAME, "salinity", "temperature", "nitrate", "ssh_adt"]
LINEAR_VARS = ["salinity", "temperature"]

In [196]:
data = (
    load_data()[REQUIRED_COLUMNS]
    .set_index(COORD_COLUMNS, drop=False)
    .pipe(drop_extreme_salinities, min=20, max=40)
    .pipe(add_talk_adjustment, fname="/Users/luke/Downloads/glodapv2_adjustments_last_updated_on_2026_07_09.csv")
    .pipe(drop_bad_quality_talk)
    .dropna(subset=NONAN_SUBSET)
    .select_dtypes(include=[np.number])
    .loc[:, FEATURE_NAMES + [TARGET_NAME]]
    .pipe(add_cyclical_dayofyear)
    .pipe(add_spherical_coords)
)

train_idx, test_idx = make_train_test_split(data, n_splits=6)
train_folds = make_train_test_folds(data.iloc[train_idx])

2026-08-14 17:31:28.332 | DEBUG    | __main__:load_data:15 - Loading 40 .pq files from ../data/training
2026-08-14 17:31:28.489 | DEBUG    | __main__:drop_bad_quality_talk:368 - TA values with large adjustments (<= 6): 2365
2026-08-14 17:31:28.489 | DEBUG    | __main__:drop_bad_quality_talk:369 - TA values without good flags (!= 2): 3046
2026-08-14 17:31:28.490 | INFO     | __main__:drop_bad_quality_talk:370 - Number of rows filtered due to large adjustments and bad flags: 5319 of 41534 (13%)
2026-08-14 17:31:28.525 | DEBUG    | __main__:make_salinity_bins:96 - Using the following bin edges for salinity: [20.1909 32.801  34.057  34.819  35.539  39.231 ]
2026-08-14 17:31:28.527 | DEBUG    | __main__:stratified_group_folds:139 - Making train-test splits stratified by salinity_bin and grouped by expocode
2026-08-14 17:31:28.595 | DEBUG    | __main__:make_salinity_bins:96 - Using the following bin edges for salinity: [20.1909 32.801  34.057  34.819  35.539  39.231 ]
2026-08-14 17:31:28.597

In [203]:
x_train = data[FEATURE_NAMES].iloc[train_idx]
x_test = data[FEATURE_NAMES].iloc[test_idx]
model = hta.BaggingCatBoostResidualRegressor(
    linear_features=["salinity"],
    feature_names=x_train.columns.tolist(),
    loss_function="RMSEWithUncertainty",
    max_samples=0.8,
    n_estimators=20,
)

In [204]:
model.fit(
    x_train,
    y=data.iloc[train_idx][TARGET_NAME],)

,n_estimators,20
,max_samples,0.8
,oob_score,False
,n_jobs,None
,random_state,None


In [205]:
model.score(
    x_test,
    y=data.iloc[test_idx][TARGET_NAME],
)

ValueError: operands could not be broadcast together with shapes (4582,) (4582,2) 

In [201]:
def make_catboost_data(data: pd.DataFrame) -> pd.DataFrame:
    features = data[FEATURE_NAMES].copy()
    datetime_features = features.select_dtypes(include=["datetime", "datetimetz"]).columns
    for feature in datetime_features:
        features[feature] = features[feature].astype("int64") / 1e9
    return features


def make_pool(
    data: pd.DataFrame, linear_model: pipeline.Pipeline, target_name: str = "talk"
) -> cb.Pool:
    """
    Create a CatBoost Pool object from the given data and linear model.

    Parameters:
    - data: pd.DataFrame containing the features and target.
    - linear_model: A fitted scikit-learn Pipeline object for linear regression.

    Returns:
    - A CatBoost Pool object.
    """
    # Predict the baseline using the linear model
    baseline = linear_model.predict(data)

    residual = data[target_name] - baseline

    # Create a CatBoost Pool with the features and baseline as labels
    pool = cb.Pool(data=make_catboost_data(data), label=residual)

    return pool


# Catboost Tuning

In [202]:
RANDOM_SEED = 42
N_TRIALS = 50
NUM_THREADS = 1
MAX_ITERATIONS = 1000
EARLY_STOPPING_ROUNDS = 50
ALWAYS_IGNORED_FEATURES = sorted(set(COORD_COLUMNS).intersection(FEATURE_NAMES))

## Feature Selection

In [90]:
outer_train_data = data.iloc[train_idx]
outer_test_data = data.iloc[test_idx]

IGNORABLE_FEATURES = sorted(set(FEATURE_NAMES) - set(ALWAYS_IGNORED_FEATURES))

In [91]:
lin_model = make_linear_pipeline(LINEAR_VARS).fit(outer_train_data[LINEAR_VARS], outer_train_data[TARGET_NAME])
lin_pred = lin_model.predict(outer_train_data[LINEAR_VARS])

resid = outer_train_data[TARGET_NAME] - lin_pred

In [100]:
NUM_FEATURES_TO_SELECT = 5

def select_features(train_data, eval_data):
    # Fit the baseline exclusively on this fold's training data.
    linear_model = make_linear_pipeline(LINEAR_VARS).fit(
        train_data,
        train_data[TARGET_NAME],
    )

    # Both pools contain residual labels calculated from the same
    # training-fitted linear model.
    train_pool = make_pool(
        train_data,
        linear_model,
        TARGET_NAME,
    )
    valid_pool = make_pool(
        eval_data,
        linear_model,
        TARGET_NAME,
    )

    selector = cb.CatBoostRegressor(
        loss_function="RMSE",
        iterations=MAX_ITERATIONS,
        random_seed=RANDOM_SEED,
        allow_writing_files=False,
        verbose=False,
        thread_count=4,
        rsm=1.0,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )

    return selector.select_features(
        train_pool,
        eval_set=valid_pool,
        features_for_select=IGNORABLE_FEATURES,
        num_features_to_select=NUM_FEATURES_TO_SELECT,
        steps=10,
        algorithm=(cb.EFeaturesSelectionAlgorithm.RecursiveByShapValues),
        shap_calc_type=cb.EShapCalcType.Regular,
        train_final_model=False,
        logging_level="Silent",
        plot=True,
    )
select_features(outer_train_data, outer_test_data)

The number of features selection steps (10) is greater than the number of features to eliminate (6). The number of steps was reduced to 6.


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

{'selected_features': [3, 4, 10, 11, 12],
 'eliminated_features_names': ['mld_dens_soda',
  'phosphate',
  'ssh_sla',
  'nitrate',
  'silicate',
  'salinity'],
 'loss_graph': {'main_indices': [0, 1, 3, 4, 4, 5],
  'removed_features_count': [0, 1, 2, 3, 4, 5, 6],
  'loss_values': [25.570915564055806,
   22.842921929475317,
   22.366402766956927,
   22.07835705701019,
   22.803360789815788,
   27.923182310729185,
   26.106053899957363]},
 'eliminated_features': [9, 8, 13, 6, 7, 5],
 'selected_features_names': ['bottomdepth',
  'temperature',
  'ice_cci',
  'chl_globcolour',
  'ssh_adt']}

## Hyperparameter tuning with nested cross-validation

Optuna tunes CatBoost on the outer training split only. Each inner fold fits its own linear baseline before training CatBoost on residuals, preventing validation-fold leakage. The objective is mean validation RMSE of the combined linear + boosted prediction.

In [28]:
from sklearn import feature_selection
from sklearn.base import BaseEstimator, RegressorMixin


class LinearCatboost(BaseEstimator, RegressorMixin):
    def __init__(self, linear_vars: list[str], catboost_params: dict):
        self.linear_vars = linear_vars
        self.linear_model = self._init_linear_model(linear_vars)
        self.catboost_params = catboost_params
        self.catboost_model = self._init_catboost_model(catboost_params)


    @staticmethod
    def _init_linear_model(linear_vars: list[str]):
        return make_linear_pipeline(linear_vars)

    @staticmethod
    def _init_catboost_model(catboost_params: dict):
        return cb.CatBoostRegressor(**catboost_params)

    def _make_pool(self, X: pd.DataFrame, y: pd.Series | None = None) -> cb.Pool:
        label = None
        if y is not None:
            label = np.asarray(y) - self.linear_model.predict(X)
        return cb.Pool(data=make_catboost_data(X), label=label)

    def fit(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        *,
        eval_set: tuple[pd.DataFrame, pd.Series] | None = None,
        **fit_params,
    ):
        self.linear_model = self._init_linear_model(self.linear_vars)
        self.linear_model.fit(X, y)
        train_pool = self._make_pool(X, y)

        eval_pool = None
        if eval_set is not None:
            X_valid, y_valid = eval_set
            eval_pool = self._make_pool(X_valid, y_valid)

        self.catboost_model = self._init_catboost_model(self.catboost_params)
        self.catboost_model.fit(train_pool, eval_set=eval_pool, **fit_params)

        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        # Predict using the linear model
        baseline = self.linear_model.predict(X)
        
        # Create a CatBoost Pool for prediction
        pool = self._make_pool(X)
        
        # Predict the residuals using the CatBoost model
        pred = self.catboost_model.predict(pool)
        if pred.ndim > 1 and pred.shape[1] > 1:
            residuals_pred = pred[:, 0]
        else:
            residuals_pred = pred
        
        # Combine baseline and residuals to get final predictions
        return baseline + residuals_pred

In [ ]:
fixed_catboost_params = {
    # "loss_function": "RMSEWithUncertainty",
    "loss_function": "RMSE",
    "iterations": MAX_ITERATIONS,
    "random_seed": RANDOM_SEED,
    "ignored_features": ALWAYS_IGNORED_FEATURES + ["mld_dens_soda", "phosphate", "ssh_sla"],
    "allow_writing_files": False,
    "verbose": False,
    "thread_count": NUM_THREADS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
}

In [ ]:
ALWAYS_IGNORED_FEATURES + ["mld_dens_soda", "phosphate", "ssh_sla"]

['lat', 'lon', 'time', 'mld_dens_soda', 'phosphate', 'ssh_sla']

In [106]:
def objective(trial: optuna.Trial) -> float:
    trial_params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 4, 9),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 100.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "rsm": trial.suggest_float("rsm", 0.6, 1.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
    }

    fold_rmse = []
    fold_best_iterations = []

    for fold_number, (fold_train_idx, fold_valid_idx) in enumerate(train_folds):
        fold_train = outer_train_data.iloc[fold_train_idx]
        fold_valid = outer_train_data.iloc[fold_valid_idx]

        fold_model = LinearCatboost(
            linear_vars=LINEAR_VARS,
            catboost_params={**fixed_catboost_params, **trial_params},
        )
        fold_model.fit(
            fold_train,
            fold_train[TARGET_NAME],
            eval_set=(fold_valid, fold_valid[TARGET_NAME]),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            use_best_model=True,
        )

        combined_prediction = fold_model.predict(fold_valid)
        score = metrics.root_mean_squared_error(
            fold_valid[TARGET_NAME], combined_prediction
        )
        fold_rmse.append(float(score))
        fold_best_iterations.append(
            fold_model.catboost_model.get_best_iteration() + 1
        )

        trial.report(float(np.mean(fold_rmse)), step=fold_number)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("fold_rmse", fold_rmse)
    trial.set_user_attr("fold_best_iterations", fold_best_iterations)
    trial.set_user_attr("refit_iterations", int(np.median(fold_best_iterations)))
    return float(np.mean(fold_rmse))

In [107]:
sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    pruner=pruner,
    storage="sqlite:///catboost_uncertainty_cv.db",
    study_name=f"catboost_uncertainty_{fixed_catboost_params['loss_function']}_ignored_features_v4",
    load_if_exists=True
)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=4, show_progress_bar=True)

[I 2026-08-14 17:06:08,600] Using an existing study with name 'catboost_uncertainty_RMSE_ignored_features_v4' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-14 17:06:29,773] Trial 53 finished with value: 23.70490725938338 and parameters: {'learning_rate': 0.017086973785292437, 'depth': 9, 'l2_leaf_reg': 2.0857297201914458, 'random_strength': 0.23874309147860306, 'rsm': 0.600239180062928, 'bagging_temperature': 0.46474642865717886}. Best is trial 45 with value: 23.57840504094012.
[I 2026-08-14 17:06:30,649] Trial 52 finished with value: 23.82898193908128 and parameters: {'learning_rate': 0.017260097875805243, 'depth': 9, 'l2_leaf_reg': 0.745180408518883, 'random_strength': 0.3028039006417951, 'rsm': 0.7789372351098329, 'bagging_temperature': 0.5099699901834942}. Best is trial 45 with value: 23.57840504094012.
[I 2026-08-14 17:06:32,244] Trial 50 finished with value: 23.913104948857516 and parameters: {'learning_rate': 0.016533313505436367, 'depth': 9, 'l2_leaf_reg': 1.917547074312073, 'random_strength': 0.07983013667609735, 'rsm': 0.7152569382878675, 'bagging_temperature': 0.4159418441888877}. Best is trial 45 with value: 23.5784

In [108]:
from optuna import visualization

cv_results = (
    study.trials_dataframe(
        attrs=("number", "value", "params", "user_attrs", "state")
    )
    .sort_values("value", na_position="last")
    .reset_index(drop=True)
)

# print(f"Best mean CV RMSE: {study.best_value:.3f}")
# print(f"Refit iterations: {study.best_trial.user_attrs['refit_iterations']}")
# cv_results.head(10)
visualization.plot_parallel_coordinate(study)

## Refit and held-out evaluation

The selected hyperparameters are refit on all outer-training observations. The held-out test split is used once for the final metrics and was not used by Optuna or early stopping.

In [109]:
lr_model = make_linear_pipeline(LINEAR_VARS).fit(
    outer_train_data, outer_train_data[TARGET_NAME]
)
train_data_boosting = make_pool(outer_train_data, lr_model, TARGET_NAME)
test_data_boosting = make_pool(outer_test_data, lr_model, TARGET_NAME)

best_trial_params = {
    name: value
    for name, value in study.best_params.items()
    if name not in {"ignored_feature_1", "ignored_feature_2"}
}
best_params = {
    **fixed_catboost_params,
    **best_trial_params,
    "iterations": study.best_trial.user_attrs["refit_iterations"],
}
boosted_trees_model = LinearCatboost(LINEAR_VARS, best_params)
boosted_trees_model.fit(outer_train_data, outer_train_data[TARGET_NAME])

,linear_vars,"['salinity', 'temperature']"
,catboost_params,"{'allow_writing_files': False, 'bagging_temperature': 0.8900264760590495, 'depth': 8, 'early_stopping_rounds': 50, ...}"


In [111]:
subset = outer_test_data
test_prediction = boosted_trees_model.predict(subset)

test_metrics = pd.Series(
    {
        "rmse": metrics.root_mean_squared_error(
            subset[TARGET_NAME], test_prediction
        ),
        "mae": metrics.mean_absolute_error(
            subset[TARGET_NAME], test_prediction
        ),
        "r2": metrics.r2_score(subset[TARGET_NAME], test_prediction),
    },
    name="held_out_test",
)
test_metrics

rmse    23.378517
mae     11.762585
r2       0.947322
Name: held_out_test, dtype: float64